In [1]:
import requests
import pandas as pd
import json
import requests_cache
import time

In [3]:
godine = range(2015, 2025)

In [11]:
# Set up cache with a 1-hour expiration
requests_cache.install_cache('f1_cache', expire_after=3600)


In [1]:
import pandas as pd

# Load CSVs
pit_stops = pd.read_csv('online_dataset/pit_stops.csv')      # raceId,driverId,stop,lap,time,duration,milliseconds
races = pd.read_csv('online_dataset/races.csv')              # raceId,year,round,...
drivers = pd.read_csv('online_dataset/drivers.csv')          # driverId,driverRef,...

# Merge pit_stops with races to get 'year' and 'round' from 'raceId'
pit_race_merged = pit_stops.merge(races[['raceId', 'year', 'round']], on='raceId', how='left')

# Merge with drivers to get 'driverRef'
final_pit_df = pit_race_merged.merge(drivers[['driverId', 'driverRef']], on='driverId', how='left')

# Filter for seasons 2018 and onward
final_pit_df = final_pit_df[final_pit_df['year'] >= 2018]

# Select and rename columns
final_pit_df = final_pit_df[['year', 'round', 'lap', 'driverRef', 'stop', 'duration', 'time']]
final_pit_df.rename(columns={'year': 'season', 'driverRef': 'driverId'}, inplace=True)

# Sort values
final_pit_df = final_pit_df.sort_values(['season', 'round', 'lap', 'driverId', 'stop'])

# Save to CSV
final_pit_df.to_csv('tables/processed_pitstop_data.csv', index=False)


In [11]:
# all_laps_data = []

# # Loop through all seasons and rounds
# for season in range(2018, 2025):
#     for round_num in range(1, 25):
#         offset = 0
#         limit = 1000  # Jolpica API max limit is 1000
#         while True:
#             url = f"https://api.jolpi.ca/ergast/f1/{season}/{round_num}/laps.json?limit={limit}&offset={offset}"
#             response = requests.get(url)
#             if response.status_code != 200:
#                 print(f"Failed: {season} Round {round_num} — Status: {response.status_code}")
#                 break

#             data = response.json()
#             races = data['MRData']['RaceTable']['Races']
#             if not races:
#                 break  # No race data

#             laps = races[0].get('Laps', [])
#             if not laps:
#                 break  # No lap data
#             len(laps)
#             for lap in laps:
#                 lap_number = int(lap['number'])
#                 for timing in lap['Timings']:
#                     all_laps_data.append({
#                         'season': season,
#                         'round': round_num,
#                         'lap': lap_number,
#                         'driverId': timing['driverId'],
#                         'position': int(timing['position']),
#                         'time': timing['time']
#                     })

#             total = int(data['MRData']['total'])
#             print(total)
#             offset += limit
#             lap_nums = [int(lap['number']) for lap in laps]
#             print(f"Laps for {season} round {round_num}: {lap_nums}")
#             if offset >= total:
#                 break


#         print(f"Got data {season}, round {round_num}")
#         time.sleep(4)  # to avoid API rate limiting




#pravi neke probleme uzecu sa neta

In [22]:
# Load CSVs
lap_times = pd.read_csv('online_dataset/lap_times.csv')
drivers = pd.read_csv('online_dataset/drivers.csv')
races = pd.read_csv('online_dataset/races.csv')

# Merge lap_times with races to get 'year' and 'round' from 'raceId'
lap_race_merged = lap_times.merge(races[['raceId', 'year', 'round']], on='raceId', how='left')

# Merge with drivers to get 'driverRef'
final_df = lap_race_merged.merge(drivers[['driverId', 'driverRef']], on='driverId', how='left')

# Filter for seasons 2018 and onward
final_df = final_df[final_df['year'] >= 2018]

# Select and rename columns
final_df = final_df[['year', 'round', 'lap', 'position', 'driverRef']]
final_df.rename(columns={'year': 'season', 'driverRef': 'driverId'}, inplace=True)

# Sort values
final_df = final_df.sort_values(['season', 'round', 'lap', 'position'])

# Save to CSV
final_df.to_csv('tables/processed_lap_data.csv', index=False)


In [ ]:
# all_laps_data = pd.DataFrame(all_laps_data)
# all_laps_data.to_csv("all_laps1.csv", index=False)
# print("Saved laps data to all_laps.csv!")


Saved laps data to all_laps.csv!


In [10]:
#izvlaci sve trke
race_list = []

# Loop over possible round numbers (up to 24 or more)
for season in godine:
    for round_num in range(1, 25):
        url = f'http://ergast.com/api/f1/{season}/{round_num}.json'
        response = requests.get(url)
        data = response.json()
        
        races = data['MRData']['RaceTable']['Races']
        if not races:
            break  # Stop when there are no more completed races
        
        race = races[0]
        race_list.append({
            'season': int(race['season']),
            'round': int(race['round']),
            'race_name': race['raceName'],
            'date': race['date'],
            'time' : race['time'],
            'circuit': race['Circuit']['circuitName'],
            'country': race['Circuit']['Location']['country']
                     })

In [11]:
# Save to CSV
df = pd.DataFrame(race_list)
df.to_csv(f'all_races.csv', index=False)
print(f'Saved completed races to all_races.csv')

Saved completed races to all_races.csv


In [ ]:
#get all circuits

url = f'https://api.jolpi.ca/ergast/f1/circuits.json?limit=1000'
response = requests.get(url)
data = response.json()

circuits = data['MRData']['CircuitTable']['Circuits']
circuits = pd.json_normalize(circuits)
all_circuits = pd.DataFrame(circuits)

all_circuits.to_csv(f'all_circuits.csv', index=False)
print(f'Saved completed races to all_circuits.csv')

In [ ]:
#get all constructors

url = f'https://api.jolpi.ca/ergast/f1/constructors.json?limit=1000'
response = requests.get(url)
data = response.json()

constructors = data['MRData']['ConstructorTable']['Constructors']
all_constructors = pd.DataFrame(constructors)

all_constructors.to_csv(f'all_constructors.csv', index=False)
print(f'Saved completed races to all_constructors.csv')

In [ ]:
#get drivers
all_drviers = pd.DataFrame()
for season in godine:
    url = f'http://ergast.com/api/f1/{season}/drivers.json'
    response = requests.get(url)
    data = response.json()

    one_race_drivers = data['MRData']['DriverTable']['Drivers']
    df = pd.DataFrame(one_race_drivers)
    all_drviers = pd.concat([all_drviers, df], ignore_index=True)


In [ ]:
all_drviers = all_drviers.drop_duplicates(subset=['code'])
all_drviers.to_csv(f'all_drivers.csv', index=False)
print(f'Saved completed races to all_drivers.csv')

In [ ]:
#izvlaci rezultate svih trka i najbrzi krug svakog od vozaca
all_results = []

for season in godine:
    for round_num in range(1, 25):
        url = f'http://ergast.com/api/f1/{season}/{round_num}/results.json?limit=1000'
        response = requests.get(url)
        data = response.json()

        races = data['MRData']['RaceTable']['Races']
        if not races:
            continue

        race = races[0]
        race_info = {
            'season': race.get('season'),
            'round': race.get('round'),
            'raceName': race.get('raceName'),
            'date': race.get('date'),
            'circuit': race.get('Circuit', {}).get('circuitName'),
        }

        for result in race.get('Results', []):
            fastest = result.get('FastestLap', {})
            time_obj = result.get('Time', {})

            row = {
                **race_info,
                'position': result.get('position'),
                'status': result.get('status'),
                'driver': result.get('Driver', {}).get('familyName'),
                'constructor': result.get('Constructor', {}).get('name'),
                'grid': result.get('grid'),
                'laps': result.get('laps'),
                'total_time': time_obj.get('time'),
                'total_time_ms': time_obj.get('millis'),
                'points': result.get('points'),

                # Fastest lap info
                'fastest_lap_rank': fastest.get('rank'),
                'fastest_lap_number': fastest.get('lap'),
                'fastest_lap_time': fastest.get('Time', {}).get('time'),
                'fastest_lap_speed': fastest.get('AverageSpeed', {}).get('speed'),
                'fastest_lap_speed_unit': fastest.get('AverageSpeed', {}).get('units'),
            }

            all_results.append(row)

In [ ]:
df = pd.DataFrame(all_results)
df.to_csv("all_race_results.csv", index=False)
print("Saved race results to all_race_results.csv")

In [88]:
#get quali results

all_qualies = []
for season in godine:
    for round in range(1, 25):
        url = f'http://api.jolpi.ca/ergast/f1/{season}/{round}/qualifying.json?limit=14000'
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Failed request: {url}")
            print(f"Status code: {response.status_code}")
            continue
        data = response.json()

        one_quali = data['MRData']['RaceTable']['Races']
        if not one_quali:
            continue
        one_quali = one_quali[0]
        race_info = {'season' : one_quali.get('season'),
            'round' : one_quali.get('round'),
            'race_name' : one_quali.get('raceName')}

        for result in one_quali.get('QualifyingResults', {}):
            row = {
                **race_info,
                'drivers_num' : result.get('number'),
                'position' : result.get('position'),
                'Q1_time' : result.get('Q1'),
                'Q2_time' : result.get('Q2'),
                'Q3_time' : result.get('Q3')
            }
            all_qualies.append(row)
        time.sleep(1)
        
        # df = pd.DataFrame(one_race_drivers)
        # all_drviers = pd.concat([all_drviers, df], ignore_index=True)

In [89]:
df = pd.DataFrame(all_qualies)
df.to_csv("all_qualies.csv", index=False)
print("Saved qualifing data to all_qualies.csv")

Saved qualifing data to all_qualies.csv


In [108]:
#rezultati sprinta

all_sprints = []
for season in range(2021, 2025):
    for round in range(1, 25):
        url = f'http://api.jolpi.ca/ergast/f1/{season}/{round}/sprint.json?limit=1000'
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Failed request: {url}")
            print(f"Status code: {response.status_code}")
            continue
        data = response.json()

        one_sprint = data['MRData']['RaceTable']['Races']
        if not one_sprint:
            continue
        one_sprint = one_sprint[0]
        race_info = {'season' : one_sprint.get('season'),
            'round' : one_sprint.get('round'),
            'race_name' : one_sprint.get('raceName')}

        for result in one_sprint.get('SprintResults', {}):
            row = {
                **race_info,
                'drivers_num' : result.get('number'),
                'position' : result.get('position'),
                'points' : result.get('points'),
                'grid' : result.get('grid'),
                'laps' : result.get('laps'),
                'status' : result.get('status'),
                'time' : result.get('Time', {}).get('time'),
                'fastest_lap' : result.get('FastestLap', {}).get('lap'),
                'fastest_laptime' : result.get('FastestLap', {}).get('Time', {}).get('time')
            }
            all_sprints.append(row)
        time.sleep(1)

In [ ]:
df = pd.DataFrame(all_sprints)
df.to_csv("all_sprints.csv", index=False)
print("Saved sprint data to all_sprints.csv")

Saved qualifing data to all_sprints.csv


In [7]:
#pitstopovi
all_pit_stops = []
for season in godine:
    for round in range(1, 25):
        url = f'http://api.jolpi.ca/ergast/f1/{season}/{round}/pitstops.json?limit=1000'
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Failed request: {url}")
            print(f"Status code: {response.status_code}")
            continue
        data = response.json()

        one_race_stops = data['MRData']['RaceTable']['Races']
        if not one_race_stops:
            continue
        one_race_stops = one_race_stops[0]
        race_info = {'season' : one_race_stops.get('season'),
            'round' : one_race_stops.get('round'),
            'race_name' : one_race_stops.get('raceName')}

        for result in one_race_stops.get('PitStops', {}):
            row = {
                **race_info,
                'driverId' : result.get('driverId'),
                'lap' : result.get('lap'),
                'stop' : result.get('stop'),
                'duration' : result.get('duration'),
                'time' : result.get('time')
                }
            all_pit_stops.append(row)
        time.sleep(1)
            


In [ ]:
df = pd.DataFrame(all_pit_stops)
df.to_csv("all_pit_stops.csv", index=False)
print("Saved pitstop data to all_pit_stops.csv")

Saved pitstop data to all_pit_stops1.csv
